## 1. DataFrame Fundamentals

In [ ]:
import pandas as pd
import sqlite3


# ---- Creating DataFrames ----
df = pd.DataFrame({
    "name": ["Asha", "Ravi", "Meena", "Karan", "Zoya"],
    "age": [25, 32, 28, 41, 36],
    "city": ["Mumbai", "Delhi", "Pune", "Mumbai", "Delhi"],
    "salary": [50000, 72000, 61000, 95000, 83000],
})
print("DataFrame from a dictionary:")
print(df)

# ---- Reading data (CSV, Excel, JSON, SQL) ----
df.to_csv("data/people.csv", index=False)       # write first, then read back
df.to_excel("data/people.xlsx", index=False)
df.to_json("data/people.json", orient="records")

csv_df  = pd.read_csv("data/people.csv")
xlsx_df = pd.read_excel("data/people.xlsx")
json_df = pd.read_json("data/people.json")

conn = sqlite3.connect(":memory:")              # small temporary database
df.to_sql("people", conn, index=False)
sql_df = pd.read_sql("SELECT * FROM people", conn)
print("\nRead from CSV / Excel / JSON / SQL ->", csv_df.shape, xlsx_df.shape, json_df.shape, sql_df.shape)

# ---- Viewing data ----
print("\nhead(2):\n", df.head(2))
print("\ntail(2):\n", df.tail(2))
print("\ninfo():")
df.info()
print("\ndescribe():\n", df.describe())

# ---- Selecting columns and rows ----
print("\nOne column:\n", df["name"])
print("\nTwo columns:\n", df[["name", "salary"]])
print("\nRow by position (iloc[0]):\n", df.iloc[0])
print("\nRows 1-2 by label (loc):\n", df.loc[1:2, ["name", "city"]])

# ---- Filtering with conditions ----
print("\nSalary > 60000:\n", df[df["salary"] > 60000])
print("\nMumbai AND age > 30:\n", df[(df["city"] == "Mumbai") & (df["age"] > 30)])
print("\nCity in list:\n", df[df["city"].isin(["Delhi", "Pune"])])

DataFrame from a dictionary:
    name  age    city  salary
0   Asha   25  Mumbai   50000
1   Ravi   32   Delhi   72000
2  Meena   28    Pune   61000
3  Karan   41  Mumbai   95000
4   Zoya   36   Delhi   83000

Read from CSV / Excel / JSON / SQL -> (5, 4) (5, 4) (5, 4) (5, 4)

head(2):
    name  age    city  salary
0  Asha   25  Mumbai   50000
1  Ravi   32   Delhi   72000

tail(2):
     name  age    city  salary
3  Karan   41  Mumbai   95000
4   Zoya   36   Delhi   83000

info():
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   name    5 non-null      str  
 1   age     5 non-null      int64
 2   city    5 non-null      str  
 3   salary  5 non-null      int64
dtypes: int64(2), str(2)
memory usage: 292.0 bytes

describe():
              age        salary
count   5.000000      5.000000
mean   32.400000  72200.000000
std     6.348228  17711.578134
min    25.000000  50000.0000

## 2. Data Cleaning

In [ ]:
import pandas as pd
import numpy as np

# A small messy DataFrame to practise on
messy = pd.DataFrame({
    " Name ": ["  alice ", "BOB", "bob", "Charlie", None, "alice "],
    "Age": ["25", "31", "31", "abc", "40", "25"],
    "City": ["mumbai", "Delhi", "Delhi", "PUNE", "Pune", "mumbai"],
    "Salary": [50000, np.nan, np.nan, 62000, 70000, 50000],
})
print("Original:\n", messy)

# Rename columns
messy = messy.rename(columns={" Name ": "name", "Age": "age", "City": "city", "Salary": "salary"})

# String operations on columns
messy["name"] = messy["name"].str.strip().str.title()   # remove spaces, Title Case
messy["city"] = messy["city"].str.strip().str.title()

# Data type conversion
messy["age"] = pd.to_numeric(messy["age"], errors="coerce")   # "abc" becomes NaN

# Handle missing values (dropna, fillna)
print("\nMissing values per column:\n", messy.isna().sum())
messy["salary"] = messy["salary"].fillna(messy["salary"].mean())   # fill with mean
messy["age"] = messy["age"].fillna(messy["age"].median())          # fill with median
messy = messy.dropna(subset=["name"])                              # drop rows with no name

# Remove duplicate
print("\nDuplicate rows:", messy.duplicated().sum())
messy = messy.drop_duplicates()

# Reset and set index
messy = messy.reset_index(drop=True)
indexed = messy.set_index("name")
print("\nCleaned:\n", messy)
print("\nWith name as index:\n", indexed)
print("\nLook up one row:\n", indexed.loc["Alice"])

Original:
       Name   Age    City   Salary
0    alice    25  mumbai  50000.0
1       BOB   31   Delhi      NaN
2       bob   31   Delhi      NaN
3   Charlie  abc    PUNE  62000.0
4       NaN   40    Pune  70000.0
5    alice    25  mumbai  50000.0

Missing values per column:
 name      1
age       1
city      0
salary    2
dtype: int64

Duplicate rows: 2

Cleaned:
       name   age    city   salary
0    Alice  25.0  Mumbai  50000.0
1      Bob  31.0   Delhi  58000.0
2  Charlie  31.0    Pune  62000.0

With name as index:
           age    city   salary
name                          
Alice    25.0  Mumbai  50000.0
Bob      31.0   Delhi  58000.0
Charlie  31.0    Pune  62000.0

Look up one row:
 age          25.0
city       Mumbai
salary    50000.0
Name: Alice, dtype: object


## 3. Data Transformation

In [3]:
import pandas as pd
import numpy as np

data = pd.DataFrame({
    "student": ["Asha", "Ravi", "Meena", "Karan", "Zoya", "Dev"],
    "subject": ["Math", "Math", "Science", "Science", "Math", "Science"],
    "marks":   [88, 45, 67, 92, 73, 55],
    "grade":   ["A", "F", "C", "A", "B", "D"],
})

# ---- Apply functions to columns ----
data["marks_plus_5"] = data["marks"].apply(lambda x: min(x + 5, 100))

def result(m):
    return "Pass" if m >= 50 else "Fail"
data["result"] = data["marks"].apply(result)

# ---- Create new columns ----
data["percentage"] = data["marks"] / 100 * 100
data["name_length"] = data["student"].str.len()

# ---- Map and replace values ----
data["grade_points"] = data["grade"].map({"A": 4, "B": 3, "C": 2, "D": 1, "F": 0})
data["subject"] = data["subject"].replace({"Math": "Mathematics"})

# ---- Sorting and ranking ----
data["rank"] = data["marks"].rank(ascending=False).astype(int)
print("Sorted by marks (high to low):\n", data.sort_values("marks", ascending=False))

# ---- Binning and categorization ----
data["band"] = pd.cut(data["marks"], bins=[0, 50, 75, 100], labels=["Low", "Medium", "High"])
data["quartile"] = pd.qcut(data["marks"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
print("\nWith new columns:\n", data)

# ---- Pivot and melt ----
scores = pd.DataFrame({
    "student": ["Asha", "Asha", "Ravi", "Ravi"],
    "subject": ["Math", "Science", "Math", "Science"],
    "marks":   [88, 79, 45, 60],
})
wide = scores.pivot(index="student", columns="subject", values="marks")   # long -> wide
print("\nPivot (long to wide):\n", wide)

long = wide.reset_index().melt(id_vars="student", var_name="subject", value_name="marks")  # wide -> long
print("\nMelt (wide to long):\n", long)

Sorted by marks (high to low):
   student      subject  marks grade  marks_plus_5 result  percentage  \
3   Karan      Science     92     A            97   Pass        92.0   
0    Asha  Mathematics     88     A            93   Pass        88.0   
4    Zoya  Mathematics     73     B            78   Pass        73.0   
2   Meena      Science     67     C            72   Pass        67.0   
5     Dev      Science     55     D            60   Pass        55.0   
1    Ravi  Mathematics     45     F            50   Fail        45.0   

   name_length  grade_points  rank  
3            5             4     1  
0            4             4     2  
4            4             3     3  
2            5             2     4  
5            3             1     5  
1            4             0     6  



With new columns:
   student      subject  marks grade  marks_plus_5 result  percentage  \
0    Asha  Mathematics     88     A            93   Pass        88.0   
1    Ravi  Mathematics     45     F            50   Fail        45.0   
2   Meena      Science     67     C            72   Pass        67.0   
3   Karan      Science     92     A            97   Pass        92.0   
4    Zoya  Mathematics     73     B            78   Pass        73.0   
5     Dev      Science     55     D            60   Pass        55.0   

   name_length  grade_points  rank    band quartile  
0            4             4     2    High       Q4  
1            4             0     6     Low       Q1  
2            5             2     4  Medium       Q2  
3            5             4     1    High       Q4  
4            4             3     3  Medium       Q3  
5            3             1     5  Medium       Q1  

Pivot (long to wide):
 subject  Math  Science
student               
Asha       88       79
Ravi

## 4. Aggregations & Grouping

In [4]:
import pandas as pd

# Real dataset: restaurant tips (244 rows)
tips = pd.read_csv("data/tips.csv")
print(tips.head(3))

# ---- groupby + aggregate functions (sum, mean, count) ----
print("\nAverage bill by day:\n", tips.groupby("day")["total_bill"].mean())
print("\nTotal tips by day:\n", tips.groupby("day")["tip"].sum())
print("\nNumber of rows by time:\n", tips.groupby("time")["tip"].count())

# ---- Multiple aggregations ----
print("\nMultiple aggregations on one column:\n",
      tips.groupby("day")["tip"].agg(["count", "mean", "min", "max"]))

print("\nDifferent aggregations per column:\n",
      tips.groupby("day").agg(
          avg_bill=("total_bill", "mean"),
          total_tip=("tip", "sum"),
          visits=("size", "count"),
      ))

# ---- Group by two columns ----
print("\nBy day and time:\n", tips.groupby(["day", "time"])["tip"].mean())

# ---- Pivot tables ----
print("\nPivot table:\n",
      pd.pivot_table(tips, values="tip", index="day", columns="time", aggfunc="mean"))

# ---- Cross tabulations ----
print("\nCrosstab (counts):\n", pd.crosstab(tips["day"], tips["smoker"]))

# ---- Transform and filter with groupby ----
# transform: keeps the same number of rows (adds group info to each row)
tips["day_avg_tip"] = tips.groupby("day")["tip"].transform("mean")
tips["tip_vs_day_avg"] = tips["tip"] - tips["day_avg_tip"]
print("\nWith transform:\n", tips[["day", "tip", "day_avg_tip", "tip_vs_day_avg"]].head())

# filter: keep only groups that meet a condition
big_days = tips.groupby("day").filter(lambda g: len(g) > 60)
print("\nDays with more than 60 rows:", big_days["day"].unique())

   total_bill   tip     sex smoker  day    time  size
0       16.99  1.01  Female     No  Sun  Dinner     2
1       10.34  1.66    Male     No  Sun  Dinner     3
2       21.01  3.50    Male     No  Sun  Dinner     3

Average bill by day:
 day
Fri     17.151579
Sat     20.441379
Sun     21.410000
Thur    17.682742
Name: total_bill, dtype: float64

Total tips by day:
 day
Fri      51.96
Sat     260.40
Sun     247.39
Thur    171.83
Name: tip, dtype: float64

Number of rows by time:
 time
Dinner    176
Lunch      68
Name: tip, dtype: int64

Multiple aggregations on one column:
       count      mean   min    max
day                               
Fri      19  2.734737  1.00   4.73
Sat      87  2.993103  1.00  10.00
Sun      76  3.255132  1.01   6.50
Thur     62  2.771452  1.25   6.70

Different aggregations per column:
        avg_bill  total_tip  visits
day                               
Fri   17.151579      51.96      19
Sat   20.441379     260.40      87
Sun   21.410000     247.39      

## 5. Merge & Join Operations

In [5]:
import pandas as pd

customers = pd.read_csv("data/customers_small.csv")   # 4 customers
orders    = pd.read_csv("data/orders_small.csv")      # 5 orders (customer 9 doesn't exist)

print("customers:\n", customers)
print("\norders:\n", orders)

# inner: only matching rows
print("\nINNER:\n", pd.merge(customers, orders, on="customer_id", how="inner"))
# left: all customers, even with no orders
print("\nLEFT:\n", pd.merge(customers, orders, on="customer_id", how="left"))
# right: all orders, even with no customer
print("\nRIGHT:\n", pd.merge(customers, orders, on="customer_id", how="right"))
# outer: everything from both
print("\nOUTER:\n", pd.merge(customers, orders, on="customer_id", how="outer"))

# join: merge using the index
print("\nJOIN (on index):\n", customers.set_index("customer_id").join(orders.set_index("customer_id"), how="inner"))

# concat: stack DataFrames on top of each other
extra = pd.DataFrame({"customer_id": [5], "name": ["Sara"], "city": ["Goa"]})
print("\nCONCAT:\n", pd.concat([customers, extra], ignore_index=True))

customers:
    customer_id   name    city
0            1    Ann  Mumbai
1            2    Raj   Delhi
2            3  Maria    Pune
3            4   John     Goa

orders:
    order_id  customer_id  amount
0       101            1     250
1       102            1     400
2       103            2     150
3       104            3     300
4       105            9     500

INNER:
    customer_id   name    city  order_id  amount
0            1    Ann  Mumbai       101     250
1            1    Ann  Mumbai       102     400
2            2    Raj   Delhi       103     150
3            3  Maria    Pune       104     300

LEFT:
    customer_id   name    city  order_id  amount
0            1    Ann  Mumbai     101.0   250.0
1            1    Ann  Mumbai     102.0   400.0
2            2    Raj   Delhi     103.0   150.0
3            3  Maria    Pune     104.0   300.0
4            4   John     Goa       NaN     NaN

RIGHT:
    customer_id   name    city  order_id  amount
0            1    Ann  Mumba